In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import nuflux
import nuSQuIDS as nsq

import propogate_flux
import utils

In [ ]:
import matplotlib
font = {
                    'serif':  'Computer Modern Roman',
#                     'weight': 200,
                    'size':   30}

matplotlib.rc('font', **font)
plt.rc('text', usetex=True)

In [ ]:
units = nsq.Const()

MUON = nsq.NeutrinoCrossSections_NeutrinoFlavor.muon
TAU = nsq.NeutrinoCrossSections_NeutrinoFlavor.tau

NEUTRINO = nsq.NeutrinoCrossSections_NeutrinoType.neutrino
ANTINEUTRINO = nsq.NeutrinoCrossSections_NeutrinoType.antineutrino

def get_flux_function(nsq_flux, flavor, rho):

    def get_flux(cz, e):

        return nsq_flux.EvalFlavor(flavor, cz, e * units.GeV, rho)

    return get_flux

In [ ]:
import matplotlib
font = {
#     'family': 'serif',
                    'serif':  'Computer Modern Roman',
                    'weight': 200,
                    'size':   30}

matplotlib.rc('font', **font)
plt.rc('text', usetex=True)

In [ ]:
Emin = 1e2
Emax = 1e5
czmin = -1.0
czmax = 0.2

cz_list = np.linspace(czmin, czmax, 101)
e_list = np.logspace(np.log10(Emin), np.log10(Emax), 101)

In [ ]:
flux = nuflux.makeFlux('honda2006')
flux.knee_reweighting_model = 'gaisserH3a_elbert'

nus_atm_pre = propogate_flux.get_nusquids_obj(3, 0., 0., 0., 0., 0., flux)

In [ ]:
sin22th24_MEOWS = 0.1
sin2th24_MEOWS = np.sqrt(sin22th24_MEOWS)
th24_MEOWS = np.arcsin(sin2th24_MEOWS)/2.
Um4sq_MEOWS = np.sin(th24_MEOWS)**2

In [ ]:
Ut4sq = 0.1

th14, th24, th34 = utils.Usq_to_thetas(Ue4sq = 0., Um4sq = Um4sq_MEOWS, Ut4sq  = Ut4sq)
print(th14, th24, th34)

In [ ]:
dm2 = 4.45 
th14 = 0
cp = 0

nus_atm_MEOWS = propogate_flux.get_nusquids_obj(4, dm2, th14, th24_MEOWS, 0., cp, flux)
nus_atm_MEOWS.EvolveState()

nus_atm_th34 = propogate_flux.get_nusquids_obj(4, dm2, th14, th24, th34, cp, flux)
nus_atm_th34.EvolveState()

In [ ]:
z_i_numu = propogate_flux.get_contour(cz_list, e_list, get_flux_function(nus_atm_pre, MUON, NEUTRINO) )
z_i_numubar = propogate_flux.get_contour(cz_list, e_list, get_flux_function(nus_atm_pre, MUON, ANTINEUTRINO) )

z_f_numu_MEOWS = propogate_flux.get_contour(cz_list, e_list, get_flux_function(nus_atm_MEOWS, MUON, NEUTRINO) )
z_f_numubar_MEOWS = propogate_flux.get_contour(cz_list, e_list, get_flux_function(nus_atm_MEOWS, MUON, ANTINEUTRINO) )

z_f_nutau_MEOWS = propogate_flux.get_contour(cz_list, e_list, get_flux_function(nus_atm_MEOWS, TAU, NEUTRINO) )
z_f_nutaubar_MEOWS = propogate_flux.get_contour(cz_list, e_list, get_flux_function(nus_atm_MEOWS, TAU, ANTINEUTRINO) )

z_f_numu_th34 = propogate_flux.get_contour(cz_list, e_list, get_flux_function(nus_atm_th34, MUON, NEUTRINO) )
z_f_numubar_th34 = propogate_flux.get_contour(cz_list, e_list, get_flux_function(nus_atm_th34, MUON, ANTINEUTRINO) )

z_f_nutau_th34 = propogate_flux.get_contour(cz_list, e_list, get_flux_function(nus_atm_th34, TAU, NEUTRINO) )
z_f_nutaubar_th34 = propogate_flux.get_contour(cz_list, e_list, get_flux_function(nus_atm_th34, TAU, ANTINEUTRINO) )

In [ ]:
fig, ax = plt.subplots(figsize = (11,9), constrained_layout=True)

ax.set_yscale("log")
ax.set_ylabel(r'E$_{\nu}^{true}$ [GeV]')
ax.set_xlabel(r'$\cos(\theta_z^{true})$')

cf = ax.contourf(cz_list, e_list, 100.*(1. - z_f_numubar_MEOWS/z_i_numubar),  levels= np.linspace(0,100,101), cmap='jet', extend='both')
cbar = fig.colorbar(cf, ax=ax, ticks = np.linspace(0.,100.,6))
cbar.set_label(r'$\bar{\nu}_\mu \to \bar{\nu}_\mu$ Disappearance [\%]')

ax.set_title(r'$\Delta m_{{41}}^2 = {dm2:.1f}~\textrm{{eV}}^{{2}}, |U_{{\mu 4}}|^{{2}} = {Um4sq:.2f}, |U_{{\tau 4}}|^{{2}} = {Ut4sq:.0f}$'.format(
    dm2 = dm2, Um4sq = Um4sq_MEOWS, Ut4sq = 0.), fontsize = 30)

for c in cf.collections:
    c.set_edgecolor("face")
    
plt.draw()
plt.savefig("numubar_Um4sq_{Um4sq:.6f}_Ut4sq_{Ut4sq:.6f}.pdf".format(Um4sq = Um4sq_MEOWS, Ut4sq=0.))
plt.show()

fig, ax = plt.subplots(figsize = (11,9), constrained_layout=True)

ax.set_yscale("log")
ax.set_ylabel(r'E$_{\nu}^{true}$ [GeV]')
ax.set_xlabel(r'$\cos(\theta_z^{true})$')

cf = ax.contourf(cz_list, e_list, 100.*(1. - z_f_numubar_th34/z_i_numubar),  levels= np.linspace(0,100,101), cmap='jet', extend='both')
cbar = fig.colorbar(cf, ax=ax, ticks = np.linspace(0.,100.,6))
cbar.set_label(r'$\bar{\nu}_\mu \to \bar{\nu}_\mu$ Disappearance [\%]')

ax.set_title(r'$\Delta m_{{41}}^2 = {dm2:.1f}~\textrm{{eV}}^{{2}}, |U_{{\mu 4}}|^{{2}} = {Um4sq:.2f}, |U_{{\tau 4}}|^{{2}} = {Ut4sq:.2f}$'.format(
    dm2 = dm2, Um4sq = Um4sq_MEOWS, Ut4sq =Ut4sq), fontsize = 30)

for c in cf.collections:
    c.set_edgecolor("face")
    
plt.draw()
plt.savefig("numubar_Um4sq_{Um4sq:.6f}_Ut4sq_{Ut4sq:.6f}.pdf".format(Um4sq = Um4sq_MEOWS, Ut4sq=Ut4sq))

plt.show()

fig, ax = plt.subplots(figsize = (11,9), constrained_layout=True)

ax.set_yscale("log")
ax.set_ylabel(r'E$_{\nu}^{true}$ [GeV]')
ax.set_xlabel(r'$\cos(\theta_z^{true})$')

cf = ax.contourf(cz_list, e_list, 100.*(z_f_nutaubar_th34/z_i_numubar),  levels= np.linspace(0,100,101), cmap='jet', extend='both')
cbar = fig.colorbar(cf, ax=ax, ticks = np.linspace(0.,100.,6))
cbar.set_label(r'$\bar{\nu}_\mu \to \bar{\nu}_\tau$ Appearance [\%]')

ax.set_title(r'$\Delta m_{{41}}^2 = {dm2:.1f}~\textrm{{eV}}^{{2}}, |U_{{\mu 4}}|^{{2}} = {Um4sq:.2f}, |U_{{\tau 4}}|^{{2}} = {Ut4sq:.2f}$'.format(
    dm2 = dm2, Um4sq = Um4sq_MEOWS, Ut4sq =Ut4sq), fontsize = 30)

for c in cf.collections:
    c.set_edgecolor("face")
    
plt.draw()
plt.savefig("nutaubar_Um4sq_{Um4sq:.6f}_Ut4sq_{Ut4sq:.6f}.pdf".format(Um4sq = Um4sq_MEOWS, Ut4sq=Ut4sq))
plt.show()